# Monte Carlo Tree earch Attempt

This notebook shall hold all the attempts for MCTS.
Given a Pauli word with k Pauli strings $P_k$, a tree can be formed where each level is at least 1 Pauli string less: $P_k \,\to\, P_{k-1}\,\to\,P_{k-2}\dots$. Each node will have as many children as it has Pauli strings. This means the number of possible actions space for each node is k. However to get for example from the root node to the child associated with its first Pauli string has $n^{n-2}$ (direct) reductions, where n is the number of nonidentity in the first Pauli string. To find the best - where best is defined as the (direct) reduction that also reduces the number of nonidentities in the Pauli word the most - reduction you would also need to traverse a tree $P^{1_n}_{k} \,\to\, P^{1_1}_{k}$ (read as the reduction of the first Pauli string from n nonidentities to 1 nonidentity). Upon the Pauli string only having 1 nonidentity, it is implementable and will be pruned from the Pauli word, producing a $P^1_{k-1}$. Because there are so many possible reductions for just one Pauli string which have their own tree a method to solve this is also necessary. A strong suggestion has just been using a heuristic way to find a good solution.

_Concerns:_ 
1. Why even have reductions for specific Pauli strings? Would not a more efficient method be trying to solve multiple Pauli strings simultaneously? - This concern is valid and is an active question. However there are no clear methods so far to solve multiple strings simultaneously which are not greedy and also can have hard claims made about them (For example with A* we have no clear metric so we cannot claim anything about optimality of our heuristic).  
2. Would not this way be unscalable with size? This remains to be seen in experiments, but a solid advantage of this is claims can be made about MCTS if it is well constructed.
3. MCTS is usually used to find the immediate best action. Our current problem requires a full path to the terminal node - How would that be done? A current idea is to have our implementation of MCTS return the path most visited if the MCTS is allowed enough runs or the path with the lowest CNOT count. How this will be done is part of the challenge of the implementation of MCTS we need. 

In [42]:
import random as rnd
import numpy as np
import math

In [36]:
# class that implements tableau and necessary operations
# References -Improved Simulation of Stabilizer Circuits by Scott Aaronson & Daniel Gottesman
# - Google Tableau implementation
class Cirq_Tableau:
    
    def __init__(
        self,
        pauli_word: list[str] = None  
    ):
        if pauli_word is None or len(pauli_word) == 0:
            self._column_num = None
            self._row_num = None

            self._ss, self._xs, self._zs = None, None, None
        else:
            if "-" in pauli_word[0]:
                self._column_num = len(pauli_word[0][1:]) 
            else:
                self._column_num = len(pauli_word[0])
            self._row_num = len(pauli_word)

            self._ss = self.create_sign(pauli_word)
            self._xs, self._zs = self.create_tab(pauli_word)

    # setters & getters 
    @property
    def ss(self) -> np.ndarray:
        return self._ss

    @ss.setter 
    def ss(self, new_ss: np.ndarray):
        self._ss = new_ss
        
    @property
    def xs(self) -> np.ndarray:
        return self._xs

    @xs.setter 
    def xs(self, new_xs: np.ndarray):
        self._xs = new_xs
        
    @property
    def zs(self) -> np.ndarray:
        return self._zs

    @zs.setter 
    def zs(self, new_zs: np.ndarray):
        self._zs = new_zs
        
    @property
    def column_num(self) -> int:
        return self._column_num

    @column_num.setter 
    def column_num(self, new_num: int):
        self._column_num = new_num
    
    @property
    def row_num(self) -> int:
        return self._row_num

    @row_num.setter 
    def row_num(self, new_num: int):
        self._row_num = new_num

    # functions that create parts of tableau
    def create_sign(self, pauli_word: list[str]):
        temp_ss = np.zeros((self.row_num), dtype=int)
        for pauli_string in pauli_word:
            if "-" in pauli_string:
                index = pauli_word.index(pauli_string)
                pauli_word[index] = pauli_string.replace("-", "")
                #print(pauli_word)
                temp_ss[index] = 1
        return temp_ss
        
    def create_tab(self, pauli_word: list[str]):
        temp_x = np.zeros((self.row_num, self.column_num), dtype=int)
        temp_z = np.zeros((self.row_num, self.column_num), dtype=int)
        for i, pauli_string in enumerate(pauli_word):
            for j, pauli in enumerate(pauli_string):
                if pauli == "X" or pauli == "Y":
                    temp_x[i][j] = 1
                if pauli == "Z" or pauli == "Y":
                    temp_z[i][j] = 1
        return temp_x, temp_z

    # Clifford operations on tableau. 
    def apply_H(self,column: int):
        self.ss ^= self.xs[:, column] & self.zs[:, column]
        self.xs[:, column], self.zs[:, column] = self.zs[:, column].copy(), self.xs[:, column].copy()

    def apply_S(self, column: int):
        self.ss ^= self.xs[:, column] & self.zs[:, column]
        self.zs[:, column] = self.xs[:, column] ^ self.zs[:, column]

    def apply_CX(self, control: int, target: int):
        self.ss ^= (
            (self.xs[:, control] & self.zs[:, target])
            &(~(self.xs[:, target] ^ self.zs[:, control]))
        )
        self.xs[:, target] ^= self.xs[:, control]
        self.zs[:, control] ^= self.zs[:, target]

    # class operations necessary for comparisons and equating
    def copy(self):
        new_tab = Cirq_Tableau()
        new_tab.column_num = self.column_num
        new_tab.row_num = self.row_num
        new_tab.ss = self.ss.copy()
        new_tab.zs = self.zs.copy()
        new_tab.xs = self.xs.copy()
        return new_tab
    
    def __eq__(self, other):
        if not isinstance(other, type(self)):
            return NotImplemented  
        return (
            self.column_num == other.column_num
            and self.row_num == other.row_num
            and np.array_equal(self.ss, other.ss)
            and np.array_equal(self.xs, other.xs)
            and np.array_equal(self.zs, other.zs)
        )
    
    def return_string(self):
        string = ''
        for i in range(self.row_num):
            if self.ss[i]:
                string += "-"  

            for j in range(self.column_num):
                if self.xs[i][j] and not self.zs[i][j]:
                    string += "X"
                elif not self.xs[i][j] and self.zs[i][j]:
                    string += "Z"
                elif self.xs[i][j] and self.zs[i][j]:
                    string += "Y"
                else:
                    string += "I"
            if i < self.row_num - 1:
                string += "\n" 

        return string
    
    def __copy__(self):
        return self.copy()
        

    def __str__(self) -> str:
        ss = np.expand_dims(self.ss, axis = 1)
        xz = np.concatenate((self.xs, self.zs, ss), axis=1)
        return str(xz)

    def __hash__(self) -> int:
        return hash(self.zs.tobytes() + self.xs.tobytes() + self.ss.tobytes())

    def __lt__(self, other):
        return self.row_num < other.row_num

## Heuristic method to solve single Pauli string

To move from $P_k \,\to\, P^\alpha_{k-1}$, where $\alpha$ is the number describing the index of the Pauli string we are reducing, we need a way a to make it deterministic. This is a necessity for MCTS. Having multiple reductions that fulfil $P_k \,\to\, P^\alpha_{k-1}$ means that there is no deterministic way to get to the child. So a heuristic way to construct this solution has been suggested 

The next three cells are used to construct the heuristic. The heuristic is broken down into three functions for readability purposes. i am sure there is a more efficient way to construct this idea or even a better heuristic generally but for now this will do

In [35]:
def identify(tableau, op_list, ctrl, targ):
    '''
    Function to identify the best operation amongst a list 

    tableau: Cirq_Tableau that holds our Pauli word
    op_list: list of action tuples which are possible operations
    ctrl: int that is index of control qubit
    targ: int that is index of target qubit
    '''
    operation = None # best operation
    benefit = -float('inf') # benefit of an operation benefit = reduce - increase 
    reduced = 0 # number of reducible pairs across the ctrl and targ (equivalent to how many single qubit operations will be removed)
    increased = 0 # number of increasing pairs across the ctrl and targ (equivalent to how many single qubit operations will be added)

    # loop to run through possible operations 
    for op in op_list:
        
        tab = tableau.copy()
        
        for act in op:
            match act[0]:
                case "H":
                    tab.apply_H(act[1])
                case "S":
                    tab.apply_S(act[1])

        # calculation of reduced and increased for this particular action
        reduce = sum((tab.xs[:, ctrl] & tab.xs[:, targ] & ~tab.zs[:, targ]) | (~tab.xs[:, ctrl] & tab.zs[:, ctrl] & tab.zs[:, targ]))
        increase = sum((~tab.xs[:, ctrl] & ~tab.zs[:, ctrl] & tab.zs[:, targ]) | (tab.xs[:, ctrl] & ~tab.xs[:, targ] & ~tab.zs[:, targ]))
        
        # condition to calculate better benefit and replace necessary values
        if reduce - increase > benefit:
            benefit = reduce - increase
            operation = op
            reduced = reduce
            increased = increased
        elif reduce - increase == benefit:
            if reduce > reduced:
                benefit = reduce - increase
                operation = op
                reduced = reduce
                increased = increase
                
    operation.append(("CX", ctrl, targ))
    return reduced, increased, operation 

In [17]:
def compare(tab, ndx, ctrl, targ):
    

    if tab.xs[ndx][ctrl] & ~tab.zs[ndx][ctrl]: # if control is X
        if tab.xs[ndx][targ] & ~tab.zs[ndx][targ]: # if target is X
            op_list = [[], [("S", ctrl)], [("H", ctrl), ("H", targ)],
                      [("H", ctrl), ("S", targ)]]
            return identify(tab, op_list, ctrl, targ)
            
        elif tab.xs[ndx][targ] & tab.zs[ndx][targ]: # if target is Y
            op_list = [[("S", ctrl), ("S", targ)], [("S", targ)], [("H", ctrl)],
                      [("H", ctrl), ("H", targ)]]
            return identify(tab, op_list, ctrl, targ)

        else:# if target is Z
            op_list = [[("S", ctrl), ("H", targ)], [("H", targ)], [("H", ctrl)],
                      [("H", ctrl), ("S", targ)]]
            return identify(tab, op_list, ctrl, targ)
            
    elif tab.xs[ndx][ctrl] & tab.zs[ndx][ctrl]: # if control is Y
        if tab.xs[ndx][targ] & ~tab.zs[ndx][targ]: # if target is X
            op_list = [[], [("S", ctrl)], [("H", ctrl)]]
            return identify(tab, op_list, ctrl, targ)
            
        elif tab.xs[ndx][targ] & tab.zs[ndx][targ]: # if target is Y
            op_list = [[("S", ctrl), ("S", targ)], [("H", ctrl),("S", targ)],
                      [("S", targ)]]
            return identify(tab, op_list, ctrl, targ)

        else:# if target is Z
            op_list = [[("H", targ)], [("S", ctrl), ("H", targ)]]
            return identify(tab, op_list, ctrl, targ)
            
    else: # if control is Z
        if tab.xs[ndx][targ] & ~tab.zs[ndx][targ]: # if target is X
            op_list = [[("H", targ)], [("H", ctrl)], [("S", ctrl),("H", targ)],
                      [("S", targ)]]
            return identify(tab, op_list, ctrl, targ)
            
        elif tab.xs[ndx][targ] & tab.zs[ndx][targ]: # if target is Y
            op_list = [[], [("S", ctrl)], [("H", targ)], [("S", ctrl),("H", targ)], [("H", ctrl), ("S", targ)]]
            return identify(tab, op_list, ctrl, targ)

        else:# if target is Z
            op_list = [[], [("H", ctrl), ("H", targ)], [("S", ctrl)], [("S", targ)], [("S", ctrl), ("S", targ)]]
            return identify(tab, op_list, ctrl, targ)

In [21]:
def heuristic(tableau, ndx):
    tab = tableau.copy()
    reduction = []
    while True:
        operation = None
        benefit = -float('inf')
        reduced = 0
        increased = 0
        
        if sum(tab.xs[ndx] | tab.zs[ndx]) == 1:
            break
            
        for i in range(tab.column_num):
            if i != tab.column_num -1 and (tab.xs[ndx][i] or tab.zs[ndx][i]):
                for j in range(i+1, tab.column_num):
                    if tab.xs[ndx][j] or tab.zs[ndx][j]:
                        reduce, increase, op = compare(tab, ndx, i, j)
                        if (reduce - increase) > benefit:
                            benefit = reduce - increase
                            operation = op
                            reduced = reduce
                            increased = increase
                        elif (reduce - increase) == benefit:
                            if reduce > reduced:
                                operation = op
                                reduced = reduce
                                increased = increase

                        reduce, increase, op = compare(tab, ndx, j, i)
                        if (reduce - increase) > benefit:
                            benefit = reduce - increase
                            operation = op
                            reduced = reduce
                            increased = increase
                        elif (reduce - increase) == benefit:
                            if reduce > reduced:
                                operation = op
                                reduced = reduce
                                increased = increase

        if operation != None:
            for action in operation:
                match action[0]:
                    case "CX":
                        tab.apply_CX(action[1], action[2])
                    case "S":
                        tab.apply_S(action[1])
                    case "H":
                        tab.apply_H(action[1])
            reduction += operation
            
    return tab, reduction

In [32]:
word = ["XXYX", "ZXYZ", "ZZYY", "XXIY"]
tb, reduction =heuristic(Cirq_Tableau(word), 0)
print(tb.return_string())
print(reduction)

-IIIX
-XXXI
-ZZIY
IIXY
[('CX', 0, 1), ('H', 0), ('H', 3), ('CX', 0, 3), ('H', 3), ('S', 2), ('CX', 3, 2)]


## Monte Carlo Tree Search Implementation

In [46]:
class Node:

    def __init__(
        self,
        state
    ):
        self._ni = 0
        self._cx_num = 0
        self._parent = None
        self._action = None
        self._children = dict()
        self._p_word = state
        

    @property
    def ni(self) -> int:
        return self._ni

    @ni.setter
    def ni(self, new_ni: int):
        self._ni = new_ni

    @property
    def cx_num(self) -> int:
        return self._cx_num

    @cx_num.setter
    def cx_num(self, new_num: int):
        self._cx_num = new_num

    @property
    def p_word(self):
        return self._ni

    @p_word.setter
    def p_word(self, new_word):
        self._p_word = new_word

    @property
    def parent(self):
        return self._parent

    @parent.setter
    def parent(self, new_parent):
        self._ni = new_parent

    @property
    def action(self):
        return self._parent

    @parent.setter
    def parent(self, new_parent):
        self._ni = new_parent
        
    @property
    def children(self):
        return self._children

    @children.setter
    def children(self, new_children):
        self._children = new_chilren

    def UCT(self, param):
        if self.parent == None:
            # it is a division by cx_num to ensure that as cx_num gets smaller we have a larger UCT
            return (1/self.cx_num)

        return (1/self.cx_num) + (param * math.sqrt(math.log(self.parent.ni)/ self.ni)) 